# 07 -- Property Value Distribution Visualization

**Phase:** Visualization
**Notebook goal:** Produce a professional, portfolio-ready horizontal bar chart showing the distribution of year-over-year assessed property value percentage changes across City of Vancouver properties.

---

## Purpose

This notebook visualizes the binned property value change distribution created in Notebook 06. The input is a small processed CSV -- `property_value_change_distribution_bins.csv` -- derived from the local row-level distribution file and already validated in the previous notebook. The raw distribution file and the full Property Tax Report are not accessed here.

The output chart is saved to `visuals/property_value_change_distribution_bins.png`.

> **Important caveat:** BC Assessment assessed values are administrative property valuations used as the basis for property tax calculations. They are not MLS sale prices, transaction prices, or market values. Year-over-year changes in assessed value should not be interpreted as direct market appreciation or depreciation. No causal claims are made between assessed value changes and housing supply or permitting activity.

## 2. Imports and Paths

We use `pathlib` for cross-platform path handling, `pandas` for loading the processed bins CSV, and `matplotlib.pyplot` for chart generation. All paths are derived from `PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT       = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
VISUALS_DIR        = PROJECT_ROOT / 'visuals'
BINS_PATH          = PROCESSED_DATA_DIR / 'property_value_change_distribution_bins.csv'
OUTPUT_PNG_PATH    = VISUALS_DIR / 'property_value_change_distribution_bins.png'

print(f'PROJECT_ROOT    : {PROJECT_ROOT}')
print(f'BINS_PATH       : {BINS_PATH}')
print(f'OUTPUT_PNG_PATH : {OUTPUT_PNG_PATH}')

## 3. Load Processed Data

We load only the small binned summary CSV -- not the raw Property Tax Report and not the large local distribution file. The CSV has 10 rows and 3 columns and loads instantly.

In [ ]:
df = pd.read_csv(BINS_PATH)

print(f'Shape   : {df.shape}')
print(f'Columns : {list(df.columns)}')
print()
display(df)

## 4. Validation Checks

Six checks confirm that the input file is the expected output from Notebook 06 before we attempt to visualise it. The checks verify file existence, non-empty content, expected column names, bucket presence, absence of duplicates, and the total property count matching the full dataset size (1,552,663 records).

In [ ]:
EXPECTED_COLS    = {'bucket', 'property_count', 'percentage_of_records'}
EXPECTED_BUCKETS = [
    'Below -50%',
    '-50% to -20%',
    '-20% to -10%',
    '-10% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 20%',
    '20% to 50%',
    'Above 50%',
    'Missing / not calculable',
]
EXPECTED_TOTAL = 1_552_663

assert BINS_PATH.exists(), f'Input file not found: {BINS_PATH}'
print('File exists                     : OK')

assert len(df) > 0, 'Input file is empty'
print(f'File is non-empty               : {len(df):,} rows -- OK')

actual_cols = set(df.columns.tolist())
assert EXPECTED_COLS == actual_cols, (
    f'Column mismatch. Expected: {sorted(EXPECTED_COLS)}. Got: {sorted(actual_cols)}'
)
print('Expected columns present        : OK')

expected_set = set(EXPECTED_BUCKETS)
actual_set   = set(df['bucket'].tolist())
assert expected_set == actual_set, (
    f'Bucket mismatch. Missing: {expected_set - actual_set}. Extra: {actual_set - expected_set}'
)
print('All expected buckets present    : OK')

assert df['bucket'].nunique() == len(df), 'Duplicate bucket names found'
print('No duplicate bucket names       : OK')

count_sum = df['property_count'].sum()
assert count_sum == EXPECTED_TOTAL, (
    f'property_count sum mismatch: {count_sum:,} != {EXPECTED_TOTAL:,}'
)
print(f'property_count sums correctly   : {count_sum:,} -- OK')

pct_sum = df['percentage_of_records'].sum()
assert abs(pct_sum - 100.0) < 0.01, (
    f'Percentages do not sum to approximately 100%: {pct_sum:.4f}'
)
print(f'Percentages sum to approx 100%  : {pct_sum:.4f}% -- OK')

## 5. Visualization

A horizontal bar chart is used because bucket labels are long strings that read naturally on the y-axis. The bucket order is preserved from the CSV (which follows the logical ordering defined in Notebook 06: most negative at the top, missing at the bottom). The y-axis is inverted so the chart reads top-to-bottom in that logical order.

Value labels are appended to the right of each bar showing the percentage rounded to one decimal place. The note below the chart states that assessed values are not MLS sale prices. No causal interpretation of the distribution is included.

In [ ]:
TITLE    = 'Distribution of Assessed Property Value Changes'
SUBTITLE = 'City of Vancouver Property Tax Report -- Percentage change buckets'
X_LABEL  = 'Share of property records (%)'
NOTE     = 'Assessed values are administrative valuations and are not MLS sale prices.'

VISUALS_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    df['bucket'],
    df['percentage_of_records'],
    color='#4C72B0',
    edgecolor='white',
    height=0.65,
)

ax.invert_yaxis()

for bar, pct in zip(bars, df['percentage_of_records']):
    ax.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        f'{pct:.1f}%',
        va='center',
        ha='left',
        fontsize=9,
        color='#333333',
    )

ax.set_xlabel(X_LABEL, fontsize=10)
ax.set_xlim(0, df['percentage_of_records'].max() * 1.22)

ax.text(0, 1.12, TITLE,    transform=ax.transAxes, fontsize=13, fontweight='bold', va='bottom')
ax.text(0, 1.05, SUBTITLE, transform=ax.transAxes, fontsize=9,  color='#666666',   va='bottom')
ax.text(0.5, -0.12, NOTE,  transform=ax.transAxes, fontsize=8,  color='#888888',   va='top', ha='center', style='italic')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_PNG_PATH, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved: {OUTPUT_PNG_PATH}')

## 6. Output Validation

Confirm the PNG was written to disk and report its file size.

In [ ]:
assert OUTPUT_PNG_PATH.exists(), f'Output PNG not found: {OUTPUT_PNG_PATH}'
print('PNG exists                      : OK')

file_size_kb = OUTPUT_PNG_PATH.stat().st_size / 1024
print(f'File size                       : {file_size_kb:.1f} KB')

## 7. Final Status

Confirm the source file, the output file, and the key constraint that this chart was generated from the small processed output rather than the raw data.

In [ ]:
print('=' * 70)
print('NOTEBOOK STATUS')
print('=' * 70)
print()
print('Input')
print(f'  File : {BINS_PATH.name}')
print(f'  Note : small processed output created from the local distribution file')
print(f'  Rows : {len(df):,} (one row per bucket)')
print()
print('Output')
print(f'  File : {OUTPUT_PNG_PATH.name}')
print(f'  Path : {OUTPUT_PNG_PATH}')
print()
print('The visualization was generated from the processed binned output.')
print('Raw data and the large local distribution file were not accessed.')
print('Assessed values are administrative property valuations, not MLS sale prices.')
print('No causal claims are made between assessed values and housing supply.')
print('=' * 70)